# Lesson 14: Heterogeneous Treatment Effects

## Opening Story: Personalized Medicine

A drug might work well for some patients but not others. Understanding who benefits most from treatment is crucial for personalized medicine and policy design.

Treatment effect heterogeneity—variation in treatment effects across individuals—is now a central focus of causal inference. Methods like CATE estimation and causal forests help us discover which subgroups benefit most.

---

## Learning Objectives

By the end of this lesson, you should be able to:

1. Define conditional average treatment effects (CATE)
2. Estimate CATE using causal forests
3. Identify subgroups with different treatment effects
4. Interpret heterogeneous treatment effects
5. Apply policy learning methods

---

## 14.1 Conditional Average Treatment Effects

### Definition

$$\tau(x) = E[Y(1) - Y(0) | X = x]$$

The treatment effect varies across subgroups defined by covariates $X$.

### Why It Matters

- **Policy targeting**: Focus resources on those who benefit most
- **Equity**: Understand if effects differ across demographics
- **Mechanism**: Discover why treatments work differently

---

## 14.2 Causal Forests

In [ ]:
import numpy as np
import pandas as pd
from econml.dml import CausalForestDML
from sklearn.ensemble import RandomForestRegressor

np.random.seed(42)
n = 2000
p = 5

# Generate data with heterogeneous effects
X = np.random.normal(0, 1, (n, p))
T = np.random.binomial(1, 0.5, n)

# Treatment effect depends on X[:, 0]
true_effect = 2 * X[:, 0]
Y = 10 + X @ np.ones(p) + true_effect * T + np.random.normal(0, 1, n)

# Fit causal forest
causal_forest = CausalForestDML(
    model_y=RandomForestRegressor(n_estimators=100, random_state=42),
    model_t=RandomForestRegressor(n_estimators=100, random_state=42),
    n_estimators=100,
    random_state=42
)

causal_forest.fit(Y, T, X=X)

# Estimate individual treatment effects
te_pred = causal_forest.effect(X)

# Compare to true effects
correlation = np.corrcoef(true_effect, te_pred)[0, 1]
print(f"Correlation between true and predicted effects: {correlation:.3f}")

# Feature importance
importances = causal_forest.feature_importances_
print(f"\nFeature importances: {importances.round(3)}")
print(f"True importance: X[:, 0] should be highest")

---

## 14.3 Policy Learning

In [ ]:
# Optimal treatment assignment
# Assign treatment to those with highest predicted effects
threshold = np.median(te_pred)
policy = (te_pred > threshold).astype(int)

# Value of policy vs. always treat
value_always = true_effect.mean()
value_policy = true_effect[policy == 1].mean() * policy.mean() + \
               0 * (1 - policy).mean()

print(f"Value of always treating: {value_always:.3f}")
print(f"Value of policy: {value_policy:.3f}")

---

## 14.4 Common Mistakes

1. **Overfitting**: Use cross-fitting
2. **Multiple testing**: Adjust for multiple comparisons
3. **Interpretation**: CATE is correlational, not necessarily causal
4. **Data splitting**: Use sample splitting for honest estimation

---

## 14.5 Knowledge Check

### Multiple Choice

1. **CATE is:**
   A) Average treatment effect for everyone
   B) Treatment effect for a specific subgroup
   C) Total treatment effect
   D) Direct treatment effect

2. **Causal forests:**
   A) Are random forests for outcomes
   B) Estimate heterogeneous treatment effects
   C) Are always better than regression
   D) Don't require assumptions

3. **Policy learning:**
   A) Determines who should be treated
   B) Estimates average effects
   C) Tests for heterogeneity
   D) All of the above

4. **Heterogeneous effects are important for:**
   A) Personalized medicine
   B) Policy targeting
   C) Understanding mechanisms
   D) All of the above

5. **Cross-fitting is used to:**
   A) Increase bias
   B) Reduce overfitting
   C) Speed up computation
   D) Increase variance

### Short Answer

6. **Explain why treatment effect heterogeneity matters for policy.**

7. **How do causal forests differ from random forests?**

8. **What is the role of cross-fitting in causal forest estimation?**

9. **How can you identify subgroups with high treatment effects?**

10. **Give an example where heterogeneous effects would change policy recommendations.**

---

## 14.6 Summary

1. **Heterogeneous effects** vary across subgroups
2. **CATE** is the key quantity of interest
3. **Causal forests** estimate CATE nonparametrically
4. **Policy learning** uses CATE for optimal treatment assignment
5. **Cross-fitting** ensures valid inference

---

## 14.7 Further Reading

- Athey, S. & Imbens, G.W. (2016). "Recursive Partitioning for Heterogeneous Causal Effects." *PNAS*.
- Wager, S. & Athey, S. (2018). "Estimation and Inference of Heterogeneous Treatment Effects using Random Forests." *JASA*.